In [12]:
# open a combined nc
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import io
from PIL import Image
import ipywidgets as widgets
import folium
import ipyleaflet

ds = xr.open_dataset(r"output\nc\LMRFC_Apr2011_combined.nc4")
ds

<xarray.Dataset> Size: 218MB
Dimensions:       (time: 384, latitude: 300, longitude: 473)
Coordinates:
  * latitude      (latitude) float64 2kB 28.53 28.57 28.6 ... 38.43 38.47 38.5
  * longitude     (longitude) float64 4kB -96.73 -96.7 -96.67 ... -81.04 -81.0
  * time          (time) datetime64[ns] 3kB 2011-04-15 ... 2011-04-30T23:00:00
Data variables:
    APCP_surface  (time, latitude, longitude) float32 218MB ...
Attributes:
    CDI:          Climate Data Interface version 1.9.5 (http://mpimet.mpg.de/...
    Conventions:  CF-1.6
    history:      Wed Mar  3 21:51:45 2021: ncks -d longitude,-96.75,-81.0 -d...
    NCO:          netCDF Operators version 4.9.1 (Homepage = http://nco.sf.ne...
    CDO:          Climate Data Operators version 1.9.5 (http://mpimet.mpg.de/...

In [7]:
# check if any gaps in time
time0 = ds.time[0] # first time
timeN = ds.time[-1] # last time
# calculate time difference based on first and last time
delta = timeN - time0
# convert to numpy timedelta64
delta = np.timedelta64(delta.values)

nsteps = ds.sizes['time'] # number of time steps in the dataset

# the number of hours between first and last time should be equal to the number of time steps - 1 in the dataset
print(delta / np.timedelta64(1, 'h'))
print((nsteps - 1) * np.timedelta64(1, 'h'))
# check if delta = nsteps - 1 hours
diff = delta - (nsteps - 1) * np.timedelta64(1, 'h')

if diff == np.timedelta64(0, 'h'):
    print("No gaps in time")
else:
    print("Gaps in time detected")

383.0
383 hours
No gaps in time


In [21]:
import base64
from io import BytesIO

# Convert PIL images to data URLs
def pil_to_data_url(img):
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    data = base64.b64encode(buffer.getvalue()).decode()
    return f"data:image/png;base64,{data}"

image_urls = [pil_to_data_url(img) for img in images]

# Create ipyleaflet map
center = (ds.latitude.mean().item(), ds.longitude.mean().item())
m_leaflet = Map(center=center, zoom=6)
# make the map larger
m_leaflet.layout.height = '600px'
m_leaflet.layout.width = '100%'

# Create initial overlay
img_overlay_leaflet = ImageOverlay(
    url=image_urls[0],
    bounds=((lat_min, lon_min), (lat_max, lon_max)),
    opacity=0.6
)
m_leaflet.add_layer(img_overlay_leaflet)

slider_leaflet = IntSlider(min=0, max=nsteps-1, step=1, value=0, description='Time')
play_leaflet = Play(
    value=0,
    min=0,
    max=nsteps-1,
    step=1,
    interval=200,
    description="Play"
)
jslink((play_leaflet, 'value'), (slider_leaflet, 'value'))
controls_leaflet = VBox([play_leaflet, slider_leaflet])

def update_overlay(change):
    m_leaflet.remove_layer(img_overlay_leaflet)
    img_overlay_leaflet.url = image_urls[change['new']]
    m_leaflet.add_layer(img_overlay_leaflet)

slider_leaflet.observe(update_overlay, names='value')

display(VBox([m_leaflet, controls_leaflet]))


In [22]:
ds.close()